In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from base_functions import *
from matplotlib.lines import Line2D
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import copy

In [ ]:
def plot_figure(number):
    cmap = get_stevepalm_cmap()

    # Defining Latitude Limits to plot
    xmin = np.min([np.min(Lat_cpl2[:,0][indices2]), 
                  np.min(Lat_EC[min_idx])])
    
    xmax = np.max([np.max(Lat_cpl2[:,0][indices2]), 
                  np.max(Lat_EC[min_idx])])
    
    plt.close('all')
    fig = plt.figure(figsize = (10, 5))
    ax2 = fig.add_subplot(211)
    data = ATB532_cpl2.T[:, indices2].flatten()
    c = ax2.pcolormesh(Lat_cpl2[:,0][indices2], Alt_cpl, 
                       ATB532_cpl2.T[:, indices2], 
                       cmap = cmap, vmin = 0, vmax = 0.03) 
                       # (0, 0.01) for Feb 04, (0, 0.015) for Feb 10
    
    print (np.quantile(data[np.isfinite(data)], 0.01),
           np.quantile(data[np.isfinite(data)], 0.99))
    
    ax2.fill_between(x = Lat_cpl2[:,0][indices2], 
                     y1 = Elv_cpl2[:,0][indices2] + 0.1,
                     y2 = 0, color = 'grey')
    
    cbar = plt.colorbar(c, extend = 'both')
    cbar.set_label(r'ATB ($km^{-1} sr^{-1}$)', fontsize = 13)
    ax2.set_ylabel('Height (km)', fontsize = 16)
    ax2.set_title('a) CPL ATB 532 nm averaged to 1 km', loc = 'left', fontsize = 16)
    #ax2.set_ylim(4, 10.) # Feb 20
    ax2.set_ylim(0, 10.) # Feb 20
    #ax2.set_ylim(0, 5.) # Feb 19
    ax2.set_xlim(xmin, xmax)
    ax2.tick_params(axis='both', which='major', labelsize=14)
    ax2.axvline(x = Lat_cpl2[:,0][closest_air_idx], color = 'red', lw = 2)
    ax2.set_xticklabels([])
    
    ax = fig.add_subplot(212)
    c = ax.pcolormesh(Lat_EC[indices_EC2], Alt_EC[:, indices_EC2], 
                       ATB355_EC[:,indices_EC2], cmap = cmap, vmin = -0.00, vmax = 0.03)
    cbar = fig.colorbar(c, extend = 'both')
    cbar.set_label(r'ATB ($km^{-1} sr^{-1}$)', fontsize = 13)
    #ax.set_ylim(4, 10.) # Feb 20
    ax.set_ylim(0, 10.) # Feb 20
    #ax.set_ylim(0, 5.) # Feb 19
    
    ax.fill_between(x = Lat_EC[indices_EC2], 
                     y1 = Elevation[indices_EC2]+0.1,
                     y2 = 0, color = 'grey')
    
    
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel('Height (km)', fontsize = 16)
    ax.set_xlabel('Latitude', fontsize = 16)
    ax.set_title('b) ATLID ATB 355 nm', fontsize = 16, loc = 'left')
    ax.axvline(x = Lat_cpl2[:,0][closest_air_idx], color = 'red', lw = 2)
    #plt.savefig('acth-comparison-feb%s.png' %day, dpi = 400)
    plt.savefig('Figure2_paperEC_%s.png' %number, dpi = 350, bbox_inches='tight')

In [ ]:
import matplotlib.patches as patches

def plot_figure_mask(number, baseline):

    # Defining Latitude Limits to plot
    xmin = np.min([np.min(Lat_cpl2[:,0][indices2]), 
                  np.min(Lat_EC[min_idx])])
    
    xmax = np.max([np.max(Lat_cpl2[:,0][indices2]), 
                  np.max(Lat_EC[min_idx])])
    
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(4, 2, width_ratios=[20, 1], wspace=0.05)  # 2nd column for colorbars
    
    ax1 = fig.add_subplot(gs[0, 0])
    cax1 = fig.add_subplot(gs[0, 1])
    
    ax2 = fig.add_subplot(gs[1, 0])
    cax2 = fig.add_subplot(gs[1, 1])

    ax3 = fig.add_subplot(gs[2, 0])
    cax3 = fig.add_subplot(gs[2, 1])
    
    ax4 = fig.add_subplot(gs[3, 0])  # no colorbar, but same width
    
    # First plot
    colors = ['#c05b16', '#386CB0', '#FFFF99', 'purple', '#BEAFD4', '#f0027f','#7FC97F']
    colors_new = []
    cmap_mio = ListedColormap(colors)
    
    c = ax1.pcolormesh(Lat_depo[indices2], Alt_cpl, featype_cpl[:, indices2],
                       cmap = cmap_mio, vmin = -1.5, vmax = 5.5)
    ax1.set_xlim(xmin, xmax)
    ax1.fill_between(x = Lat_cpl2[:,0][indices2], 
                     y1 = Elv_cpl2[:,0][indices2] + 0.1,
                     y2 = 0, color = '#c05b16')
    
    # Colorbar first plot
    
    cbar1 = fig.colorbar(c, cax=cax1, ticks=[-1, 0, 1, 2, 3, 4, 5])
    cbar1.ax.set_yticklabels(['Surface', 'Invalid', 'Water', 'Undetermined cloud type', 'Ice', 'Attenuated', 'Aerosol']) 
    
    ax1.set_ylabel('Height (km)', fontsize = 16)
    ax1.set_title('a) CPL Level 2 Feature Type + Cloud phase', loc = 'left', fontsize = 16)
    ax1.set_ylim(0, 11)
    #ax1.set_ylim(4, 10)
    ax1.set_xlim(xmin, xmax)
    ax1.tick_params(axis='both', which='major', labelsize=14)
    ax1.set_xticklabels([])
    
    # Second plot
    
    c = ax2.pcolormesh(Lat_EC_ba[indices_EC2_ba], Alt_EC_ba.T[indices_EC2_ba, :].T, 
                       mask_feature_ba.T[indices_EC2_ba, :].T, 
                       #mask_feature.where(qc <= 1,  np.nan).T[indices_EC2, :].T,
                       cmap = 'Accent_r',  vmin = -3.5, vmax = 4.5)
    
    #ax2.plot(tropo_latitude, tropo_height/1000., c = 'k', zorder = 2)
    
    # color bar second plot
    cbar2 = fig.colorbar(c, cax=cax2, ticks=[-3, -2, -1, 0, 1, 2, 3, 4])
    cbar2.ax.set_yticklabels(['Missing', 'Surface', 'Attenuated', 'Clear', 'Water', 'Supercooled', 'Ice', 'Aerosol']) 
    ax2.set_ylim(0, 11)
    #ax2.set_ylim(4, 10)
    ax2.set_xlim(xmin, xmax)
    ax2.tick_params(axis='both', which='major', labelsize=14)
    ax2.set_xticklabels([])
    #ax2.set_xlabel('Latitude', fontsize = 16)
    
    ax2.set_title('b) ATLID baseline BA Level 2a classification' , loc = 'left', fontsize = 16)
    ax2.set_ylabel('Height (km)', fontsize = 16)

    
    # Third plot

    c = ax3.pcolormesh(Lat_EC[indices_EC2], Alt_EC.T[indices_EC2, :].T, 
                       mask_feature.T[indices_EC2, :].T, 
                       #mask_feature.where(qc <= 1,  np.nan).T[indices_EC2, :].T,
                       cmap = 'Accent_r',  vmin = -3.5, vmax = 4.5)
    
    #ax2.plot(tropo_latitude, tropo_height/1000., c = 'k', zorder = 2)
    
    # color bar second plot
    cbar2 = fig.colorbar(c, cax=cax3, ticks=[-3, -2, -1, 0, 1, 2, 3, 4])
    cbar2.ax.set_yticklabels(['Missing', 'Surface', 'Attenuated', 'Clear', 'Water', 'Supercooled', 'Ice', 'Aerosol']) 
    ax3.set_ylim(0, 11)
    #ax2.set_ylim(4, 10)
    ax3.set_xlim(xmin, xmax)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    ax3.set_xticklabels([])
    #ax2.set_xlabel('Latitude', fontsize = 16)
    
    ax3.set_title('c) ATLID baseline BD Level 2a classification', loc = 'left', fontsize = 16)
    ax3.set_ylabel('Height (km)', fontsize = 16)

    # Four plot
    
    ax4.contour(Lat_depo[indices2], Alt_cpl[:], borders != 0,
                levels=[0.5], colors='#1E88E5', linewidths=1)
    
    #ax3.contour(Lat_EC[indices_EC2], Alt_EC.T[indices_EC2, :].T, borders_ec != 0,
    #            levels=[0.5], colors='#D81B60', linewidths=2)

    lat_1d = Lat_EC[indices_EC2]                    # shape (236,)
    lat_1d_ba = Lat_EC_ba[indices_EC2_ba]                    # shape (236,)
    alt_2d = Alt_EC.T[indices_EC2, :].T             # shape (242, 236)
    alt_2d_ba = Alt_EC_ba.T[indices_EC2_ba, :].T             # shape (242, 236)
    
    # broadcast lat_1d across rows so it matches alt_2d's shape
    lat_2d = np.broadcast_to(lat_1d[np.newaxis, :], alt_2d.shape)  # shape (242, 236)
    lat_2d_ba = np.broadcast_to(lat_1d_ba[np.newaxis, :], alt_2d_ba.shape)  # shape (242, 236)
    
    ax4.contour(lat_2d, alt_2d, borders_ec != 0,
                levels=[0.5], colors='#004D40', linewidths=1)

    ax4.contour(lat_2d_ba, alt_2d_ba, borders_ec_ba != 0,
                levels=[0.5], colors='#D81B60', linewidths=1)

    custom_lines = [Line2D([0], [0], color='#D81B60', lw=3),
                    Line2D([0], [0], color='#004D40', lw=3),
                    Line2D([0], [0], color='#1E88E5', lw=3),]
                    #Line2D([0], [0], color='#D81B60', lw=1.5, ls = '--')]
    ax4.legend(custom_lines, ['ATLID BA ice cloud',  'ATLID BD ice cloud', 'CPL ice cloud'], #'ATLID total cloud'], 
                    fontsize=12.4, loc = 2, ncol = 3)
    #ec_top['ice-base'].plot(ax = ax3, color = 'red')
    #cpl_top['ice-base'].plot(ax = ax3, color = 'navy')
    
    # Location of box
    y0 = Alt_EC.T[0, 173].values
    y1 = Alt_EC.T[0, 154].values
    x0 = Lat_EC[indices_EC2][160]
    x1 = Lat_EC[indices_EC2][0]
    
    ax4.set_xlabel('Latitude', fontsize = 16)
    ax4.set_ylabel('Height (km)', fontsize = 16)
    ax4.tick_params(axis='both', which='major', labelsize=14)
    ax4.set_xlim(xmin, xmax)
    #ax4.set_ylim(0, 11)
    ax4.set_ylim(2, 9)
    #ax4.set_ylim(5, 10)
    ax4.set_title('d) Comparison CPL vs ATLID cloud borders', loc = 'left', fontsize = 16)
    
    plt.savefig('Figure3_paperEC_%s.png' %(number), dpi = 350, bbox_inches='tight')

In [ ]:
def select_files(baseline, day):
    if baseline == 'CA':
        # Reading EarthCare files data using Joe's class level 2
        path_ebd = './path_ebd/' # Enter your path here
        filepath04  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250220T214303Z_20250721T124721Z_04169D.h5'
        
        # Reading EarthCare files data using Joe's class level 2 A-TC
        path_atc = './path_atc/' # Enter your path here
        filepath04_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250204T213904Z_20260909T183408Z_03920D.h5'
        filepath07_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250207T220907Z_20260909T184744Z_03967D.h5'
        filepath10_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250210T223848Z_20260909T184856Z_04014D.h5'
        filepath12_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250212T222744Z_20260909T185011Z_04045D.h5'
        filepath19_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250219T223450Z_20260909T185134Z_04154D.h5'
        filepath20_atc  = path_atc + 'ECA_EXAA_ATL_TC__2A_20250220T214303Z_20260909T185312Z_04169D.h5'
    
    if baseline == 'BA':
        # Reading EarthCare files data using Joe's class level 2
        path_ebd = './path_ebd/' # Enter your path here
        filepath04  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20  = path_ebd + 'ECA_EXBA_ATL_EBD_2A_20250220T214303Z_20250721T124721Z_04169D.h5'
        
        # Reading EarthCare files data using Joe's class level 2 A-TC
        path_atc = './path_atc/' # Enter your path here
        filepath04_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250204T213904Z_20250721T124157Z_03920D.h5'
        filepath07_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250207T220907Z_20250721T124431Z_03967D.h5'
        filepath10_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250210T223848Z_20250912T012831Z_04014D.h5'
        filepath12_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250212T222744Z_20250721T124522Z_04045D.h5'
        filepath19_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250219T223450Z_20250721T124618Z_04154D.h5'
        filepath20_atc  = path_atc + 'ECA_EXBA_ATL_TC__2A_20250220T214303Z_20250721T124721Z_04169D.h5'

    files_dict = {'4':  [filepath04, filepath04_atc],
                  '7':  [filepath07, filepath07_atc],
                  '10': [filepath10, filepath10_atc],
                  '12': [filepath12, filepath12_atc], 
                  '19': [filepath19, filepath19_atc], 
                  '20': [filepath20, filepath20_atc]}
    
    return files_dict[day]

# Reading CPL processed data
path = './path/' # Enter your path here
fileday04 = path + 'ATB_GLOVE_04feb25_merra-2.hdf5'
fileday07 = path + 'ATB_GLOVE_07feb25_merra-2.hdf5'
fileday10 = path + 'ATB_GLOVE_10feb25_merra-2.hdf5'
fileday12 = path + 'ATB_GLOVE_12feb25_merra-2.hdf5'
fileday19 = path + 'ATB_GLOVE_19feb25_merra-2.hdf5'
fileday20 = path + 'ATB_GLOVE_20feb25_merra-2.hdf5'

# Reading CPL feature mask
# New processed Level 2 Layer data
path = './path/' # Enter your path here
fileday_fe04 = path + 'CPL_L2_V1-02_01kmPro_259013_04feb25.hdf5'
fileday_fe07 = path + 'CPL_L2_V1-02_01kmPro_259014_07feb25.hdf5'
fileday_fe10 = path + 'CPL_L2_V1-02_01kmPro_259015_10feb25.hdf5'
fileday_fe12 = path + 'CPL_L2_V1-02_01kmPro_259016_12feb25.hdf5'
fileday_fe19 = path + 'CPL_L2_V1-02_01kmPro_259018_19feb25.hdf5'
fileday_fe20 = path + 'CPL_L2_V1-02_01kmPro_259019_20feb25.hdf5'

cpl_dict = {'4':  [fileday04, fileday_fe04],
            '7':  [fileday07, fileday_fe07],
            '10': [fileday10, fileday_fe10],
            '12': [fileday12, fileday_fe12], 
            '19': [fileday19, fileday_fe19], 
            '20': [fileday20, fileday_fe20]}

titles = ['04', '07', '10', '12', '19', '20'] 

In [ ]:
for i, day in enumerate(['7']): #, '7']): #, '10', '12', '19', '20']:

    filepath, filepath_atc = select_files(baseline = 'CA', day = day)
    fileday, fileday_fe = cpl_dict[day]
    
    date_results = extract_and_organize_data(filepath_ec = filepath,
                                         filepath_ec_atc = filepath_atc,
                                        fileday_cpl = fileday, 
                                        fileday_cpl_fe = fileday_fe, 
                                        range_height = True, layer1 = 0, layer2 = 13, minutos = 10)
                                        #range_height = True, layer1 = 6.6, layer2 = 8.7, minutos = 10)
    Lat_cpl2         = date_results[0]
    Lat_EC           = date_results[1]
    ATB532_cpl2      = date_results[2]
    indices2         = date_results[3]
    Alt_cpl          = date_results[4]
    Elv_cpl2         = date_results[5]
    closest_air_idx  = date_results[6]
    Alt_EC           = date_results[7]
    indices_EC2      = date_results[9]
    min_idx          = date_results[10]
    ATB355_EC        = date_results[11]
    Elevation        = date_results[12]
    Lat_depo         = date_results[13]
    featype_cpl      = date_results[14]
    mask_feature     = date_results[15]

    lat_1d = Lat_EC[indices_EC2]                    # shape (236,)
    alt_2d = Alt_EC.T[indices_EC2, :].T             # shape (242, 236)

    # Computing the borders
    borders = featype_cpl[:, indices2].copy()
    borders = np.where(np.isin(borders, [1, 2, 3]), borders, 0)
    #borders[borders != 3] = 0
    
    borders_ec =  mask_feature.T[indices_EC2, :].T.data.copy()
    borders_ec = np.where(np.isin(borders_ec, [1, 2, 3]), borders_ec, 0)
    #borders_ec[borders_ec != 3] = 0

    # BORDER BA
    filepath_ba, filepath_atc_ba = select_files(baseline = 'BA', day = day)
    fileday, fileday_fe = cpl_dict[day]
    
    date_results_ba = extract_and_organize_data(filepath_ec = filepath_ba,
                                         filepath_ec_atc = filepath_atc_ba,
                                        fileday_cpl = fileday, 
                                        fileday_cpl_fe = fileday_fe, 
                                        range_height = True, layer1 = 0, layer2 = 13, minutos = 10)
    
    Lat_cpl2_ba         = date_results_ba[0]
    Lat_EC_ba           = date_results_ba[1]
    ATB532_cpl2_ba      = date_results_ba[2]
    indices2_ba         = date_results_ba[3]
    Alt_cpl_ba          = date_results_ba[4]
    Elv_cpl2_ba         = date_results_ba[5]
    closest_air_idx_ba  = date_results_ba[6]
    Alt_EC_ba           = date_results_ba[7]
    indices_EC2_ba      = date_results_ba[9]
    min_idx_ba          = date_results_ba[10]
    ATB355_EC_ba        = date_results_ba[11]
    Elevation_ba        = date_results_ba[12]
    Lat_depo_ba         = date_results_ba[13]
    featype_cpl_ba      = date_results_ba[14]
    mask_feature_ba     = date_results_ba[15]

    
    borders_ec_ba =  mask_feature_ba.T[indices_EC2_ba, :].T.data.copy()
    borders_ec_ba = np.where(np.isin(borders_ec_ba, [1, 2, 3]), borders_ec_ba, 0)

    plot_figure(titles[i])
    plot_figure_mask(titles[i], baseline = 'BD')
    